# Lab 3 bridge: from Brevitas to hls4ml
**AIMS Edge ML for Science · Day 6 · ~20 minutes · works on its own**

The hls4ml tutorial starts from Keras models. Our models are PyTorch + Brevitas. This notebook is the **plumbing** between the two: it takes a quantized Brevitas model,
turns it into an hls4ml FPGA design, checks that the design computes **exactly** the same thing, and shows what hls4ml built. Section 3 shows the same steps for any model of yours,
which is what capstone teams with an FPGA target will use.

No FPGA tools are needed: hls4ml compiles a bit-accurate C++ emulation of the design right here in Colab.

> **Before you start:** **File → Save a copy in Drive**, then run the cells in order (Shift + Enter). The first cell installs hls4ml and friends (~2 min).

## 0. Setup and a quantized model

In [ ]:
# --- Colab setup: run once per session (~2 min: installs the hls4ml toolchain) ---
import os, sys, glob, subprocess
IN_COLAB = "google.colab" in sys.modules
USE_DRIVE = False       # True: keep the dataset on your Google Drive between sessions
LAB = "lab03_hls4ml"
if IN_COLAB:
    ROOT = "/content/aims-fastml-labs"
    if not os.path.exists(ROOT):
        bundles = glob.glob("/content/*bundle*.zip")
        if bundles:
            subprocess.run(["unzip", "-q", "-o", bundles[0], "-d", ROOT], check=True)
        else:
            subprocess.run(["git", "clone", "-q", "https://github.com/SagarA17/aims-fastml-labs.git", ROOT], check=True)
    else:
        subprocess.run(["git", "-C", ROOT, "pull", "-q"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "hls4ml==1.3.0", "quantizers<2", "brevitas", "qonnx", "onnx", "onnxoptimizer"], check=True)
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.environ["FASTML_DATA"] = "/content/drive/MyDrive/fastml_data"
    os.chdir(f"{ROOT}/{LAB}")
print("working directory:", os.getcwd())

In [ ]:
import time, warnings
warnings.filterwarnings("ignore")
sys.path.append("..")
import numpy as np, torch, pandas as pd, matplotlib.pyplot as plt
from common.jets import load_jets, CLASSES
from common.compress import mlp, qat_mlp, train, accuracy, bkg_rejection, ptq, bops
from common.hls import to_hls, hls_predict, check_bit_exact, dense_summary

X_train, X_test, y_train, y_test, _ = load_jets()
idx = np.random.default_rng(0).permutation(len(X_train))[:200_000]
Xtr, ytr = torch.from_numpy(X_train[idx]), torch.from_numpy(y_train[idx])
Xte, yte = torch.from_numpy(X_test[:20_000]), torch.from_numpy(y_test[:20_000])
CAL = Xtr[:20_000]
TOP = CLASSES.index("t")

def rejection_from_scores(scores, y, eff=0.5):
    """Top-tag background rejection at 50% efficiency, from raw model outputs (works for PyTorch and for the FPGA emulation)."""
    p = torch.softmax(torch.as_tensor(scores, dtype=torch.float32), 1)[:, TOP].numpy(); y = np.asarray(y)
    cut = np.quantile(p[y == TOP], 1 - eff); fpr = (p[y != TOP] >= cut).mean()
    return float("inf") if fpr == 0 else 1 / fpr

# The model you saved at the end of Lab 2 (on your Drive), if you have it; otherwise train a 6-bit QAT model now (~1 min).
model, origin = None, ""
if IN_COLAB:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
for path in ["/content/drive/MyDrive/edgeml_models/lab2_model.pt", "../lab02_quantization/my_models/lab2_model.pt"]:
    if os.path.exists(path):
        ck = torch.load(path, weights_only=False)
        model = qat_mlp(ck["cfg"], ck["sizes"]); model.load_state_dict(ck["state_dict"]); model.eval(); origin = f"loaded from {path}"
        break
if model is None:
    model = train(qat_mlp([(6, 6)] * 4), Xtr, ytr, epochs=3, seed=0); origin = "trained just now (6-bit QAT)"
print(f"model {origin} | accuracy {accuracy(model, Xte, yte):.4f} | top-tag rejection {bkg_rejection(model, Xte, yte, sig=TOP):.0f}")

## 1. From PyTorch to an FPGA design (10 min)
`to_hls` exports the Brevitas model (via the QONNX format), lets hls4ml turn it into a C++/HLS design, and **compiles a bit-accurate emulation** of that design.

*Two details are handled for you inside `common/hls.py`:* the biases are quantized during training (so hls4ml doesn't have to guess their type), and the emulation is fed inputs already
rounded by the model's input quantizer (so they aren't rounded twice). With both in place, the FPGA design computes **exactly** what PyTorch computes.

In [ ]:
t0 = time.perf_counter()
hm, _ = to_hls(model, output_dir="hls_prj_rf1", reuse_factor=1)
maxdiff, agree = check_bit_exact(hm, model, Xte[:5000].numpy())
scores_fpga = hls_predict(hm, model, Xte.numpy())
with torch.no_grad():
    scores_torch = model(Xte).numpy()
print(f"converted + compiled in {time.perf_counter()-t0:.0f} s | max |PyTorch − FPGA emulation| = {maxdiff} | same prediction for {agree:.1%} of jets")
print(f"physics, PyTorch:        accuracy {(scores_torch.argmax(1) == yte.numpy()).mean():.4f} | rejection {rejection_from_scores(scores_torch, yte):.0f}")
print(f"physics, FPGA emulation: accuracy {(scores_fpga.argmax(1) == yte.numpy()).mean():.4f} | rejection {rejection_from_scores(scores_fpga, yte):.0f}")

📌 **What this shows.** The difference is exactly zero: the FPGA design is not an approximation of your model, it *is* your model. So everything you measured in PyTorch
(accuracy, rejection) holds on the hardware, and you only have to worry about whether the design *fits*.

## 2. What hls4ml built (10 min)
For each layer: the number type of its weights and of its running sum (accumulator), how many weights are exactly zero, and how many multipliers it needs.

In [ ]:
pd.DataFrame(dense_summary(hm))

📌 **What this shows.**
- **The accumulators are much wider than the weights.** Sums of many products need extra integer bits, and the bias brings its own fractional bits (Day 3).
  Low precision makes the stored numbers and the multipliers cheap; the adders stay wide.
- **Some weights are exactly zero even though you never pruned.** At low bit widths, small weights round to zero. On an FPGA those multipliers are simply not built: quantization prunes a little, for free.

## 3. The same steps for your own model
Any model built with `qat_mlp` (or quantized with `ptq`) from `common/compress.py` converts the same way. The three lines you need:

In [ ]:
my_model = model                                                     # <- any qat_mlp / ptq model of yours
hm_mine, _ = to_hls(my_model, output_dir="hls_prj_mine", reuse_factor=1)  # build + compile the emulation
print("exact match:", check_bit_exact(hm_mine, my_model, Xte[:2000].numpy()))  # (max difference, agreement): expect (0.0, 1.0)

**Capstone teams with an FPGA target:** once a design is ready, `make_submission(...)` from `common/submit.py` packages it for the instructor's synthesis runs (see `capstone/targets.md`).

**Optional, if you have Vitis HLS** (most of you won't): `hm.build(csim=False, synth=True)` runs real synthesis, and `hls4ml.report.read_vivado_report("hls_prj_rf1")` prints the
actual resources and latency. The instructor's reports for a few designs are in `reports/`.